# Modelo

In [1]:
import numpy as np
import pandas as pd
import joblib
import os

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import randint, uniform, norm, loguniform

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV

from config import SEED

Elegimos dataset

In [2]:
dataset = 'ciclos_r_hl256_rms'

## Espectrogramas

Se suelen usar Mel espectrogramas

In [3]:
train_data = np.load(f'./dataset/{dataset}/train_melspectrogram.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/{dataset}/test_melspectrogram.npz')
X_test = test_data['X']
y_test = test_data['y']

In [4]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((9843, 47616), (9843,), (1296, 47616), (1296,))

### Random Forest

#### Entrenamiento

In [36]:
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, max_features='sqrt')
param_distributions = {
    'max_depth': randint(8, 20),
    'min_samples_split': randint(5, 20),
    'min_samples_leaf': randint(5, 20)
}

In [37]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=30,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV 1/5] END max_depth=14, min_samples_leaf=8, min_samples_split=17;, score=0.697 total time=  51.4s
[CV 2/5] END max_depth=14, min_samples_leaf=8, min_samples_split=17;, score=0.713 total time=  34.6s
[CV 3/5] END max_depth=14, min_samples_leaf=8, min_samples_split=17;, score=0.687 total time=  45.2s
[CV 4/5] END max_depth=14, min_samples_leaf=8, min_samples_split=17;, score=0.692 total time=  29.7s
[CV 5/5] END max_depth=14, min_samples_leaf=8, min_samples_split=17;, score=0.673 total time=  28.0s
[CV 1/5] END max_depth=18, min_samples_leaf=12, min_samples_split=17;, score=0.701 total time=  28.4s
[CV 2/5] END max_depth=18, min_samples_leaf=12, min_samples_split=17;, score=0.720 total time=  30.0s
[CV 3/5] END max_depth=18, min_samples_leaf=12, min_samples_split=17;, score=0.683 total time=  27.7s
[CV 4/5] END max_depth=18, min_samples_leaf=12, min_samples_split=17;, score=0.682 total time=  27.7s
[CV 5/5] END max_depth=18

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....0023FBD8BB770>, 'min_samples_leaf': <scipy.stats....0023FBD4474D0>, 'min_samples_split': <scipy.stats....0023FBD4475D0>}"
,n_iter,30
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [38]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
26,15,8,6,0.696177
11,19,13,5,0.693338
0,14,8,17,0.692352
4,18,12,9,0.692094
1,18,12,17,0.692094
10,17,10,17,0.691971
13,17,16,16,0.691943
23,14,16,17,0.691930
21,12,6,8,0.691654
20,17,13,14,0.691394


In [39]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 15, 'min_samples_leaf': 8, 'min_samples_split': 6}
Best CV score: 0.6961768197200706


In [40]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.98      0.97      5024
           1       0.98      0.96      0.97      4819

    accuracy                           0.97      9843
   macro avg       0.97      0.97      0.97      9843
weighted avg       0.97      0.97      0.97      9843



#### Evaluación

In [41]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.69      0.63      0.66       663
           1       0.64      0.70      0.67       633

    accuracy                           0.67      1296
   macro avg       0.67      0.67      0.66      1296
weighted avg       0.67      0.67      0.66      1296



#### Guardado

In [42]:
os.makedirs(f'./modelos/{dataset}', exist_ok=True)

joblib.dump(best_model, f'./modelos/{dataset}/melspec_rf.pkl')

['./modelos/ciclos_r_hl256_rms/melspec_rf.pkl']

### Gradient Boost

In [5]:
gbc = GradientBoostingClassifier(
    n_estimators=1000,
    n_iter_no_change=10,
    max_features='sqrt',
    random_state=SEED
    )

In [51]:
param_grid = {
    'learning_rate': [0.03, 0.05, 0.07, 0.1]
}

rnd_search = GridSearchCV(
    estimator=gbc,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV 1/5] END ................learning_rate=0.03;, score=0.690 total time= 2.6min
[CV 2/5] END ................learning_rate=0.03;, score=0.695 total time=  47.2s
[CV 3/5] END ................learning_rate=0.03;, score=0.647 total time=  22.4s
[CV 4/5] END ................learning_rate=0.03;, score=0.647 total time=  43.1s
[CV 5/5] END ................learning_rate=0.03;, score=0.661 total time= 1.0min
[CV 1/5] END ................learning_rate=0.05;, score=0.681 total time= 1.1min
[CV 2/5] END ................learning_rate=0.05;, score=0.694 total time=  35.2s
[CV 3/5] END ................learning_rate=0.05;, score=0.659 total time=  27.8s
[CV 4/5] END ................learning_rate=0.05;, score=0.673 total time=  54.8s
[CV 5/5] END ................learning_rate=0.05;, score=0.668 total time=  43.8s
[CV 1/5] END ................learning_rate=0.07;, score=0.680 total time= 1.1min
[CV 2/5] END ................learning_rate=0.07;,

,estimator,GradientBoost...ndom_state=42)
,param_grid,"{'learning_rate': [0.03, 0.05, ...]}"
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


In [52]:
pd.DataFrame(rnd_search.cv_results_)[['param_learning_rate', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_learning_rate,mean_test_score
1,0.05,0.675030
3,0.10,0.671997
2,0.07,0.670785
0,0.03,0.667886


In [53]:
best_model = rnd_search.best_estimator_

In [7]:
best_model.n_estimators_

106

In [8]:
predict_train = best_model.predict(X_train)
print(classification_report(y_train, predict_train))

              precision    recall  f1-score   support

           0       0.70      0.67      0.68      5024
           1       0.67      0.69      0.68      4819

    accuracy                           0.68      9843
   macro avg       0.68      0.68      0.68      9843
weighted avg       0.68      0.68      0.68      9843



In [9]:
predict_test = best_model.predict(X_test)
print(classification_report(y_test, predict_test))

              precision    recall  f1-score   support

           0       0.68      0.62      0.65       663
           1       0.64      0.70      0.67       633

    accuracy                           0.66      1296
   macro avg       0.66      0.66      0.66      1296
weighted avg       0.66      0.66      0.66      1296



In [11]:
joblib.dump(best_model, f'./modelos/{dataset}/melspec_gbc.pkl')

['./modelos/ciclos_r_hl256_rms/melspec_gbc.pkl']

## Features de Audio

In [58]:
train_data = np.load(f'./dataset/{dataset}/train_features.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/{dataset}/test_features.npz')
X_test = test_data['X']
y_test = test_data['y']

In [59]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((9843, 46), (9843,), (1296, 46), (1296,))

### Random Forest

#### Entrenamiento

In [63]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_distributions = {
    'max_depth': randint(13, 20),
    'min_samples_split': randint(15, 30),
    'min_samples_leaf': randint(5, 15),
    'max_features': ['sqrt', 'log2', None]
}

In [64]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV 1/5] END max_depth=19, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.726 total time=   0.7s
[CV 2/5] END max_depth=19, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.752 total time=   0.7s
[CV 3/5] END max_depth=19, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.735 total time=   0.7s
[CV 4/5] END max_depth=19, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.743 total time=   0.7s
[CV 5/5] END max_depth=19, max_features=sqrt, min_samples_leaf=12, min_samples_split=27;, score=0.733 total time=   0.7s
[CV 1/5] END max_depth=17, max_features=None, min_samples_leaf=14, min_samples_split=17;, score=0.727 total time=   4.4s
[CV 2/5] END max_depth=17, max_features=None, min_samples_leaf=14, min_samples_split=17;, score=0.754 total time=   4.8s
[CV 3/5] END max_depth=17, max_features=None, min_samples_leaf=14, min_samples_split=17;, s

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....0023FC6DF3620>, 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': <scipy.stats....0023FC6DF3AF0>, 'min_samples_split': <scipy.stats....0023FC6DF38C0>}"
,n_iter,50
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [65]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
41,19,7,20,0.748294
26,16,7,15,0.748172
24,19,5,23,0.745377
27,15,5,25,0.745104
9,19,7,19,0.744867
15,17,7,28,0.744537
40,19,8,27,0.744339
8,16,7,26,0.744286
5,16,5,26,0.744102
3,16,10,19,0.743674


In [66]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 19, 'max_features': None, 'min_samples_leaf': 7, 'min_samples_split': 20}
Best CV score: 0.7482936466228021


In [67]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95      5024
           1       0.96      0.95      0.95      4819

    accuracy                           0.95      9843
   macro avg       0.95      0.95      0.95      9843
weighted avg       0.95      0.95      0.95      9843



#### Evaluación

In [68]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.76      0.78      0.77       663
           1       0.77      0.75      0.76       633

    accuracy                           0.76      1296
   macro avg       0.76      0.76      0.76      1296
weighted avg       0.76      0.76      0.76      1296



#### Guardado

In [69]:
joblib.dump(best_model, f'./modelos/{dataset}/features_rf.pkl')

['./modelos/ciclos_r_hl256_rms/features_rf.pkl']

### Gradient Boosting

In [71]:
gbc = GradientBoostingClassifier(
    n_estimators=1000,
    n_iter_no_change=10,
    random_state=SEED
    )

In [80]:
param_grid = {
    'learning_rate': [0.1, 0.15, 0.2]
}

rnd_search = GridSearchCV(
    estimator=gbc,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 3 candidates, totalling 15 fits
[CV 1/5] END .................learning_rate=0.1;, score=0.699 total time=  27.4s
[CV 2/5] END .................learning_rate=0.1;, score=0.710 total time=  19.9s
[CV 3/5] END .................learning_rate=0.1;, score=0.709 total time=  22.9s
[CV 4/5] END .................learning_rate=0.1;, score=0.711 total time=  19.3s
[CV 5/5] END .................learning_rate=0.1;, score=0.694 total time=  13.2s
[CV 1/5] END ................learning_rate=0.15;, score=0.686 total time=  12.7s
[CV 2/5] END ................learning_rate=0.15;, score=0.706 total time=   9.0s
[CV 3/5] END ................learning_rate=0.15;, score=0.677 total time=   4.8s
[CV 4/5] END ................learning_rate=0.15;, score=0.709 total time=  14.7s
[CV 5/5] END ................learning_rate=0.15;, score=0.704 total time=  16.9s
[CV 1/5] END .................learning_rate=0.2;, score=0.662 total time=   3.9s
[CV 2/5] END .................learning_rate=0.2;,

,estimator,GradientBoost...ndom_state=42)
,param_grid,"{'learning_rate': [0.1, 0.15, ...]}"
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


In [81]:
pd.DataFrame(rnd_search.cv_results_)[['param_learning_rate', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_learning_rate,mean_test_score
0,0.10,0.704778
1,0.15,0.696589
2,0.20,0.695239


In [82]:
best_model = rnd_search.best_estimator_

In [83]:
best_model.n_estimators_

156

In [84]:
predict_train = best_model.predict(X_train)
print(classification_report(y_train, predict_train))

              precision    recall  f1-score   support

           0       0.75      0.73      0.74      5024
           1       0.73      0.75      0.74      4819

    accuracy                           0.74      9843
   macro avg       0.74      0.74      0.74      9843
weighted avg       0.74      0.74      0.74      9843



In [85]:
predict_test = best_model.predict(X_test)
print(classification_report(y_test, predict_test))

              precision    recall  f1-score   support

           0       0.72      0.69      0.70       663
           1       0.68      0.71      0.70       633

    accuracy                           0.70      1296
   macro avg       0.70      0.70      0.70      1296
weighted avg       0.70      0.70      0.70      1296



In [86]:
joblib.dump(best_model, f'./modelos/{dataset}/features_gbc.pkl')

['./modelos/ciclos_r_hl256_rms/features_gbc.pkl']